# Lecture 10 · Notebook 3 — Speech as tokens: neural codecs and speech recognition
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture10/lecture10_3_speech_as_tokens.ipynb)

**Companion to [Part 5 of the video lecture](https://nlp-course-cuhksz.github.io/lecture10.html#p5-s84)** (slides 84–88). To let a language model *speak*, we first turn audio into
discrete tokens. Here you will:

1. compress speech with a **neural codec** (EnCodec) into a grid of tokens and decode it back,
2. hear how quality changes with the number of codebooks (the bitrate),
3. check with **Whisper** whether the words survive compression, and
4. compare how many tokens speech needs per second with how many a transcript needs.

Runs on CPU in a few minutes; a GPU is faster.

In [ ]:
!pip -q install -U transformers

In [ ]:
import torch, torchaudio, requests, io
import soundfile as sf
from IPython.display import Audio, display
from transformers import EncodecModel, AutoProcessor, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
codec = EncodecModel.from_pretrained("facebook/encodec_24khz").to(device).eval()
codec_proc = AutoProcessor.from_pretrained("facebook/encodec_24khz")

wav, sr = sf.read(io.BytesIO(requests.get("https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac", timeout=30).content))
wav = torch.tensor(wav, dtype=torch.float32)
if wav.ndim > 1:
    wav = wav.mean(dim=1)
wav24 = torchaudio.functional.resample(wav, sr, 24000)
print(f"{len(wav24) / 24000:.1f} s of speech")
display(Audio(wav24.numpy(), rate=24000))

## 1 · Encode speech into tokens
EnCodec runs at **75 frames per second**. Each frame is described by several codebooks (residual vector quantization): the first
codebook captures the coarse sound, later ones add detail. More codebooks = higher bitrate = better audio.

In [ ]:
@torch.no_grad()
def codec_roundtrip(bandwidth):
    x = codec_proc(raw_audio=wav24.numpy(), sampling_rate=24000, return_tensors="pt").to(device)
    enc = codec.encode(x["input_values"], x["padding_mask"], bandwidth=bandwidth)
    codes = enc.audio_codes                         # [chunks, batch, n_codebooks, frames]
    audio = codec.decode(enc.audio_codes, enc.audio_scales, x["padding_mask"])[0]
    return codes, audio[0, 0].cpu().numpy()

for bw in [1.5, 3.0, 6.0, 12.0]:
    codes, rec = codec_roundtrip(bw)
    nq, frames = codes.shape[-2], codes.shape[-1]
    sec = len(wav24) / 24000
    print(f"{bw:4.1f} kbps: {nq:2d} codebooks x {frames} frames = {nq * frames} tokens ({nq * frames / sec:.0f} tokens/s)")
    display(Audio(rec, rate=24000))

## 2 · Do the words survive? Ask Whisper
Run speech recognition on the original and on each reconstruction. Semantic content often survives even at low bitrate, while voice
quality degrades — one reason models separate *semantic* and *acoustic* tokens (slide 86).

In [ ]:
asr = pipeline("automatic-speech-recognition", model="openai/whisper-small", device=0 if device == "cuda" else -1)
def transcribe(a):
    a16 = torchaudio.functional.resample(torch.tensor(a), 24000, 16000).numpy()
    return asr({"raw": a16, "sampling_rate": 16000})["text"].strip()

print("original :", transcribe(wav24.numpy()))
for bw in [1.5, 6.0]:
    print(f"{bw:4.1f} kbps:", transcribe(codec_roundtrip(bw)[1]))

## 3 · Speech tokens vs. text tokens
The lecture's key point (slide 80): one second of speech carries 2–3 words, but a codec needs dozens to hundreds of tokens for it.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("gpt2")
text = transcribe(wav24.numpy())
sec = len(wav24) / 24000
text_tokens = len(tok(text)["input_ids"])
codes, _ = codec_roundtrip(6.0)
speech_tokens = codes.shape[-2] * codes.shape[-1]
print(f"text  : {text_tokens} tokens  ({text_tokens / sec:.1f} per second)")
print(f"speech: {speech_tokens} tokens ({speech_tokens / sec:.0f} per second at 6 kbps) -> {speech_tokens / text_tokens:.0f}x more")

## Exercises
1. Record yourself (any phone recording as WAV/MP3, upload it in the *Files* panel) and run the same analysis.
2. At which bandwidth can you no longer recognise the speaker's voice? At which one does Whisper start making mistakes?
3. Low-frame-rate tokenizers such as Mimi (Moshi) run at 12.5 Hz. How many tokens per second would 8 codebooks at 12.5 Hz be, compared with EnCodec?
4. Why would a speech LLM generate the *text* of its answer first and the *speech tokens* second (slide 88)?

**Back to the [video lecture]({SITE})**